In [1]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [2]:
p_dir = project_dir()
cleaned_dir_path = p_dir.CLEANED_DIR
featured_dir_path = p_dir.FEATURE_ENGINEERED

In [3]:
featured_dir_path

PosixPath('/home/arson/birdy/amit/noCartInsights/data/feature_engineered')

In [4]:
order_items_f = featured_dir_path/'featured_order_items.csv'
orders_f = featured_dir_path/'featured_orders.csv'
products_f = featured_dir_path/'featured_products.csv'

In [5]:
reviews = cleaned_dir_path/'cleaned_order_reviews.csv'
reviews_df = pd.read_csv(reviews,index_col=0)
reviews_df[['order_id','review_id']].duplicated().sum()

np.int64(0)

In [6]:
order_items_f_df = pd.read_csv(order_items_f,index_col=0)
order_items_f_df.sample(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_amount,seller_revenue,seller_order_count
32767,4a473906f11c8fe0749d47d07c24d13f,2,b8fefc9c1f951969968e904d08bfa8cd,da8622b14eb17ae2831f4ac5b9dab84a,2018-02-23 20:10:25,79.9,9.77,89.67,185192.32,1551
26443,3c2efaf967ce7ac93a892e7fd066e837,2,b6911a73311462653bc381da86a3a3c6,f8db351d8c4c4c22c6835c19a46f01b0,2017-08-30 16:55:14,16.9,8.27,25.17,63079.80,724
36781,538c572a252f5c6e5cc6a9f172c47a0e,1,7fc9ddb575be2fd4783d8a2207f664b0,6d1b9c9579132c87d2703ec38c30f2c5,2017-07-06 12:45:13,79.9,19.80,99.70,367.39,4


## T-Test hypothesis

### Do delayed orders receive significantly lower review scores compared to orders delivered on time?

##### To perform this hypothesis, we need order_id, is_delayed and reviews_score of an order. but we is_delayed column in orders_df and reviews_score in reviews dataframe, however, we have multiple reviews for the same orders in our reviews dataframe, so we are gonna take the mean the score for those case

In [7]:
reviews_df.info()

<class 'pandas.DataFrame'>
Index: 98410 entries, 0 to 99223
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                98410 non-null  str  
 1   order_id                 98410 non-null  str  
 2   review_score             98410 non-null  int64
 3   review_comment_title     98410 non-null  str  
 4   review_comment_message   98410 non-null  str  
 5   review_creation_date     98410 non-null  str  
 6   review_answer_timestamp  98410 non-null  str  
 7   has_comment              98410 non-null  bool 
dtypes: bool(1), int64(1), str(6)
memory usage: 6.1 MB


In [8]:
reviews_df[reviews_df['order_id'].duplicated()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment
1119,46abf3ea0b2710ad41390fdb79c32d84,5040757d4e06a4be96d3827b860b4e7c,5,No Comment,No Comment,2017-11-07 00:00:00,2017-11-10 20:07:48,True
8108,40294ea5a778dc62080d6b3f55d361ce,e1bc1083cd7acd30d0576335373b907d,5,No Comment,No Comment,2018-03-23 00:00:00,2018-03-24 00:23:06,True
11807,2af839ae66a65959d6ae07775d0e7a35,bd859dd7a6a1b8c1df9fcb75c3604eaf,3,No Comment,bom,2018-03-28 00:00:00,2018-04-11 20:20:55,True
12594,bc85f39adbafaddfda29d372f3825873,f63a31c3349b87273468ff7e66852056,5,No Comment,No Comment,2018-01-11 00:00:00,2018-01-12 02:05:02,True
15921,d5bf42808fd0df2b766d5c8ece19b3b6,8ff88873f03f1912c00741f8e5ae6c79,5,No Comment,No Comment,2018-02-01 00:00:00,2018-02-05 18:51:59,True
...,...,...,...,...,...,...,...,...
98612,d23bba9a2f1d16e5505a02e5968c1e68,19fe6cd13dca5943f17abd2c37c46abd,5,No Comment,No Comment,2017-09-15 00:00:00,2017-09-22 16:39:24,True
98654,e28cc2a1bf48c11dbcc990894356bd82,1de86d094f7dd41cca13d246d3b7fd07,5,No Comment,No Comment,2017-11-15 00:00:00,2017-11-17 05:12:37,True
98677,8ae90d960cb871f44ebf423568e2985d,baed56f3eda9223b74c6cf175f05678e,5,No Comment,No Comment,2018-04-10 00:00:00,2018-04-10 19:10:35,True
98768,9c6dc4a9e9d3532bf73335c908a8b9be,f2f99bdf2e5cc73abc5e135a2ab1767e,5,No Comment,EU RECOMENDO SIM!\r\n,2018-04-03 00:00:00,2018-04-10 14:38:27,True


In [9]:
reviews_df['has_comment'].value_counts()

has_comment
True    98410
Name: count, dtype: int64

In [10]:
reviews_mean_df = reviews_df.groupby('order_id').agg(
    reviews_mean = ('review_score','mean')
)


In [11]:
reviews_df.shape

(98410, 8)

In [12]:
orders_f_df = pd.read_csv(orders_f)
orders_f_df.sample(3)

,order_id,customer_id,order_status,order_purchase,approved_at,delivered_carrier_date,delivered_customer_date,estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed,customer_unique_id,customer_order_count,is_repeat_customer,order_value,order_count,total_spending_by_cust,avg_order_value_count
38106,2dfceb6a6d47a3835536db64f6c610a7,d4aa842316467bde3b41ece4b5698704,delivered,2018-02-26 19:56:58,2018-02-26 20:15:20,2018-02-27 23:08:34,2018-03-01 15:57:33,2018-03-16,2.0,-15.0,0.0,52ed57870beadf4e3f065252fc4fadcc,1,0,82.62,1.0,82.62,82.62
2353,daa1c13dd8df972cbc97ca351c67f32b,71e8560996119b529ab63d5a963f1a89,delivered,2018-07-31 20:39:04,2018-08-01 13:31:39,2018-08-02 17:46:00,2018-08-24 21:18:22,2018-09-12,23.0,-19.0,0.0,0af856b84f956b11317832a27b18ff4b,1,0,391.08,2.0,391.08,391.08
7181,371dbb83f599fc061ccd8558b577499d,4734f4333bded5aa9666a99377672440,delivered,2017-11-18 18:56:44,2017-11-18 19:06:25,2017-11-20 22:27:00,2017-11-29 18:48:54,2017-12-12,10.0,-13.0,0.0,95e3c8fd4233b735843cfa9f150ce487,1,0,190.37,1.0,190.37,190.37


In [25]:
t_test_df = reviews_mean_df.merge(
    orders_f_df[['order_id','is_delayed']],
    on='order_id',
    how='left'
)

In [27]:
t_test_df.sample(5)

,order_id,reviews_mean,is_delayed
57801,97b7decf3d24b8e8db00cab51c75c372,5.0,0.0
24291,3f905b9ea57c91684a194b1815fb513b,5.0,0.0
87021,e29877e3cc2991e3842d677d5e865a09,5.0,0.0
11305,1d85312d372335db21f7f178c2325c6c,5.0,0.0
5539,0e6b2d0fe443a6d38c0f6447f4eb2262,5.0,NaN


In [15]:
orders_f_df['is_delayed'].value_counts()

is_delayed
0.0    88652
1.0     7826
Name: count, dtype: int64

In [43]:
orders_f_df.loc[
    orders_f_df['order_id']=='0e6b2d0fe443a6d38c0f6447f4eb2262'
]

,order_id,customer_id,order_status,order_purchase,approved_at,delivered_carrier_date,delivered_customer_date,estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed,customer_unique_id,customer_order_count,is_repeat_customer,order_value,order_count,total_spending_by_cust,avg_order_value_count
55515,0e6b2d0fe443a6d38c0f6447f4eb2262,35bfa7ce75679810efa7d433811c6945,unavailable,2017-11-03 20:44:04,2017-11-07 08:30:54,NaN,NaN,2017-11-28,NaN,NaN,NaN,bb0aea12660bf8bd536b39fddbc336ee,2,1,NaN,NaN,95.67,95.67


In [46]:
t_test_df['is_delayed'].isnull().sum()

np.int64(2791)

#### for our hypothesis, we need only those orders that are delivered to customers, not the NaN, these are orders that were never delivered to the customers, 
---> there are 2791 rows or entries that were never delivered to the customers, so we have to drop those rows. 

In [ ]:
t_test_df['is_delayed'] = t_test_df['is_delayed'].dropna()

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
98123    0.0
98124    0.0
98125    0.0
98126    0.0
98127    0.0
Name: is_delayed, Length: 95337, dtype: float64

In [48]:
t_test_df['is_delayed'].isnull().sum()

np.int64(2791)